# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mahmoud-Beram/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: The paper claims that longer articles (higher word count) get more traffic. My Methodology Question: Did the validation design check if this relationship holds when using medians instead of means? A few extremely long, viral articles can heavily skew the mean, creating a false directional claim that length always equals traffic.

Finding 2: The paper suggests that older articles naturally decay and lose traffic over time (a content lifecycle). My Methodology Question: Where does the label come from? Is this a true lifecycle tracking the exact same cohort of articles over time, or is it measuring only the survivors? If we only measure articles that survived until today, we introduce survivor bias and distort the actual age curve.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
# Section 2: Before/After Honest Split Analysis
from huggingface_hub import hf_hub_download
from google.colab import userdata
import duckdb
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

hf_token = userdata.get('HF_TOKEN').strip()
local_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    repo_type="dataset",
    token=hf_token
)
con = duckdb.connect()

# Include client_hash_id this time so we can group by it!
df = con.execute(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) as total_imp,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_pos
    FROM '{local_file}'
    GROUP BY client_hash_id, content_hash_id
    HAVING total_imp >= 500
""").df()

df['actual_ctr'] = df['total_clicks'] / df['total_imp']

X = df[['avg_pos', 'total_imp']]
y = df['actual_ctr']
groups = df['client_hash_id']

# --- BEFORE: Random Split (Week 5 - Overconfident) ---
X_train_rnd, X_test_rnd, y_train_rnd, y_test_rnd = train_test_split(X, y, test_size=0.2, random_state=42)
rf_rnd = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf_rnd.fit(X_train_rnd, y_train_rnd)
score_rnd = rf_rnd.score(X_test_rnd, y_test_rnd)

# --- AFTER: Honest Grouped Split (Week 6 - Strict Reality) ---
# The model will train on some clients, and test on completely UNSEEN clients!
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, X_test_grp = X.iloc[train_idx], X.iloc[test_idx]
y_train_grp, y_test_grp = y.iloc[train_idx], y.iloc[test_idx]

rf_grp = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf_grp.fit(X_train_grp, y_train_grp)
score_grp = rf_grp.score(X_test_grp, y_test_grp)

print(f"Random Split R^2 Score (Overconfident): {score_rnd:.4f}")
print(f"Grouped Split R^2 Score (Honest Reality): {score_grp:.4f}")


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Random Split R^2 Score (Overconfident): 0.0603
Grouped Split R^2 Score (Honest Reality): -0.0810


In [2]:
# --- Feature Engineering: Adding Client Baseline ---
# Calculate the overall average CTR for each client
client_baselines = df.groupby('client_hash_id').apply(
    lambda x: x['total_clicks'].sum() / x['total_imp'].sum()
).reset_index(name='client_avg_ctr')

# Merge this new feature back into our main dataset
df_improved = df.merge(client_baselines, on='client_hash_id', how='left')

# Our new features (X) now include the client's baseline!
X_new = df_improved[['avg_pos', 'total_imp', 'client_avg_ctr']]
y_new = df_improved['actual_ctr']
groups_new = df_improved['client_hash_id']

# --- The Honest Grouped Split AGAIN ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X_new, y_new, groups_new))

X_train_grp_new, X_test_grp_new = X_new.iloc[train_idx], X_new.iloc[test_idx]
y_train_grp_new, y_test_grp_new = y_new.iloc[train_idx], y_new.iloc[test_idx]

rf_grp_new = RandomForestRegressor(n_estimators=50, max_depth=5, random_state=42)
rf_grp_new.fit(X_train_grp_new, y_train_grp_new)
score_grp_new = rf_grp_new.score(X_test_grp_new, y_test_grp_new)

print(f"Old Grouped Split R^2 (Without Client Baseline): {score_grp:.4f}")
print(f"NEW Grouped Split R^2 (With Client Baseline): {score_grp_new:.4f}")


Old Grouped Split R^2 (Without Client Baseline): -0.0810
NEW Grouped Split R^2 (With Client Baseline): -0.1586


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.